# Data Case Study Template

**Author:** Your Name  
**Date:** YYYY-MM-DD  
**Audience:** Hiring manager / stakeholder / internal team

Use this notebook as a reusable structure for analytics case studies. Replace the placeholder text with the real business context, your hypotheses, and your final recommendations.


## 1. Business Context

**Problem statement**  
Describe the business problem in 2 to 4 sentences.

**Decision this analysis should support**  
State the decision a stakeholder could make after reading this notebook.

**Primary metric**  
Example: conversion rate, retention, revenue per user, churn rate.

**Secondary metrics**  
Example: sessions, basket size, time on site, CAC, average order value.

**Scope and assumptions**
- Time period covered by the dataset
- Population included and excluded
- Important assumptions or known data limitations


## 2. Analytical Questions

List the concrete questions you want to answer.

1. Which customer segments convert best?
2. Which acquisition channels drive the strongest outcomes?
3. What user behaviors are most associated with conversion?
4. What actions should the business prioritize next?


## 3. Setup

Before running this notebook in VS Code:
1. Create and activate the virtual environment
2. Install packages from `requirements.txt`
3. Select the project kernel in the notebook

The detailed steps are in `README.md`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
plt.style.use('ggplot')


In [ ]:
# Change this if you want to use a different CSV file.
DATA_PATH = Path('data.csv')

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Put your CSV in the project root or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df)} rows and {df.shape[1]} columns from {DATA_PATH.resolve()}')
display(df.head())


## 4. Dataset Overview

**Source**  
Describe where the data came from.

**Grain**  
What does one row represent?

**Important fields**  
Call out the columns that matter most for your analysis.


In [ ]:
overview = pd.DataFrame(
    {
        'column': df.columns,
        'dtype': df.dtypes.astype(str).values,
        'missing_values': df.isna().sum().values,
        'missing_pct': (df.isna().mean() * 100).round(1).values,
        'unique_values': df.nunique(dropna=False).values,
    }
)

display(overview)
display(df.describe(include='all').transpose())


## 5. Data Quality Check

Document issues before analysis so the reader knows what you trust and what you do not.

Checklist:
- Missing values
- Duplicate rows
- Unexpected categories
- Outliers or suspicious ranges
- Date parsing issues


In [ ]:
quality_report = {
    'rows': len(df),
    'columns': df.shape[1],
    'duplicate_rows': int(df.duplicated().sum()),
    'total_missing_values': int(df.isna().sum().sum()),
}

display(pd.Series(quality_report, name='value').to_frame())
display(df.isna().sum().sort_values(ascending=False).to_frame('missing_values'))


## 6. Cleaning and Feature Engineering

Record each transformation so your work stays reproducible.

Suggested notes:
- Which columns were cast to date or numeric types
- Whether duplicates were removed
- How nulls were handled
- What derived features were created


In [ ]:
analysis_df = df.copy()

if 'signup_date' in analysis_df.columns:
    analysis_df['signup_date'] = pd.to_datetime(analysis_df['signup_date'], errors='coerce')

analysis_df = analysis_df.drop_duplicates().copy()

if {'sessions', 'pages_viewed'} <= set(analysis_df.columns):
    analysis_df['pages_per_session'] = np.where(
        analysis_df['sessions'] > 0,
        analysis_df['pages_viewed'] / analysis_df['sessions'],
        np.nan,
    )

if 'order_value' in analysis_df.columns:
    analysis_df['is_purchaser'] = analysis_df['order_value'].fillna(0).gt(0)

display(analysis_df.head())
print(f'Analysis dataset shape: {analysis_df.shape}')


## 7. Exploratory Data Analysis

Use this section to understand the data before drawing conclusions.

Recommended flow:
1. Distribution of core numeric metrics
2. Breakdown by segment or channel
3. Relationships between engagement and outcomes
4. Time trends if dates are available


In [ ]:
numeric_cols = analysis_df.select_dtypes(include=np.number).columns.tolist()
if numeric_cols:
    analysis_df[numeric_cols].hist(figsize=(14, 10), bins=10)
    plt.suptitle('Numeric Feature Distributions', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No numeric columns available for histograms.')


In [ ]:
if {'segment', 'customer_id', 'converted', 'order_value', 'sessions'} <= set(analysis_df.columns):
    segment_summary = (
        analysis_df.groupby('segment', dropna=False)
        .agg(
            customers=('customer_id', 'count'),
            conversion_rate=('converted', 'mean'),
            avg_order_value=('order_value', 'mean'),
            avg_sessions=('sessions', 'mean'),
        )
        .sort_values('conversion_rate', ascending=False)
    )
    display(segment_summary)

    segment_summary['conversion_rate'].plot(kind='bar', figsize=(8, 4), title='Conversion Rate by Segment')
    plt.ylabel('Conversion Rate')
    plt.tight_layout()
    plt.show()
else:
    print('Expected segment-level columns were not found.')


In [ ]:
if {'channel', 'customer_id', 'converted', 'time_on_site_min', 'order_value'} <= set(analysis_df.columns):
    channel_summary = (
        analysis_df.groupby('channel', dropna=False)
        .agg(
            customers=('customer_id', 'count'),
            conversion_rate=('converted', 'mean'),
            avg_time_on_site=('time_on_site_min', 'mean'),
            avg_order_value=('order_value', 'mean'),
        )
        .sort_values('conversion_rate', ascending=False)
    )
    display(channel_summary)

    channel_summary['avg_order_value'].plot(kind='bar', figsize=(8, 4), title='Average Order Value by Channel')
    plt.ylabel('Average Order Value')
    plt.tight_layout()
    plt.show()
else:
    print('Expected channel-level columns were not found.')


## 8. Hypotheses and Deep Dives

Write down the hypotheses before interpreting the charts.

Example hypotheses:
- H1: Higher engagement is associated with higher conversion.
- H2: Some channels drive higher-quality traffic than others.
- H3: Segment differences are large enough to justify targeted actions.


In [ ]:
selected_cols = [
    col
    for col in ['sessions', 'time_on_site_min', 'pages_viewed', 'added_to_cart', 'converted', 'order_value']
    if col in analysis_df.columns
]

if selected_cols:
    corr = analysis_df[selected_cols].corr(numeric_only=True)
    display(corr)

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(corr.index)))
    ax.set_yticklabels(corr.index)
    ax.set_title('Correlation Matrix for Key Metrics')
    fig.colorbar(im)
    plt.tight_layout()
    plt.show()
else:
    print('No columns available for correlation analysis.')


## 9. Insight Summary

Turn the analysis into concise findings. A strong pattern is:

- **What happened:** the factual observation
- **Why it matters:** the business implication
- **Evidence:** the metric, chart, or comparison supporting it

Example:
- Segment A converts 2x better than Segment B, which suggests targeted budget allocation could improve efficiency.


## 10. Recommendations

Write recommendations in stakeholder language.

Template:
1. **Recommendation:** What should the team do?
2. **Expected impact:** Which metric should improve?
3. **Confidence level:** High / Medium / Low
4. **Evidence:** Which findings support this recommendation?
5. **Tradeoff or risk:** What should the team monitor?


## 11. Limitations and Next Steps

Be explicit about what this notebook does not prove.

- Correlation does not imply causation
- Small sample sizes may make some comparisons unstable
- Missing product, geography, or cohort fields may limit the analysis
- Further validation may require experimentation or a larger dataset

**Suggested next steps**
- Run an A/B test or quasi-experimental follow-up
- Add more historical data
- Introduce cohort, funnel, or retention analysis
- Build a lightweight model only if prediction is part of the business problem
